In [1]:
# ============================================
# 1. ИМПОРТ НЕОБХОДИМЫХ БИБЛИОТЕК
# ============================================

import os
import glob
import numpy as np
import yaml
from ultralytics import YOLO
import onnx
import onnxruntime as ort
from onnx import helper, numpy_helper, shape_inference

In [9]:
# ================
# 2. ВЫБОР МОДЕЛИ
# ================

MODEL_PATH = 'models/YOLOv8n_augmented/weights/best.pt'
DATASET_YAML_PATH = 'models/dataset_augmented.yaml'
ONNX_WEIGHTS_PATH = 'models/YOLOv8n_augmented/weights/export/best_int8_quantized_forced_int8_cubeai.onnx'

In [3]:
# ================================
# 3. КВАНТИЗАЦИЯ И ЭКСПОРТ МОДЕЛИ
# ================================

def load_yaml_config(yaml_path):
    """Загружает конфигурацию модели из YAML файла"""
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

def create_custom_yolo_model(model_path, yaml_config_path):
    """Создает кастомизированную YOLO модель из YAML конфига"""
    print("Создание кастомизированной модели из YAML...")

    # Загружаем конфиг
    config = load_yaml_config(yaml_config_path)

    # Создаем модель из конфига
    model = YOLO(model_path)  # Базовая архитектура

    return model

def quantize_custom_yolo_to_int8(model_path, yaml_config_path=None, imgsz=640):
    """
    Полная квантизация кастомизированной YOLOv8 модели в INT8
    Особенно оптимизировано для моделей с уменьшенными каналами
    """

    print("="*70)
    print("ПОЛНАЯ КВАНТИЗАЦИЯ КАСТОМИЗИРОВАННОЙ YOLOv8 МОДЕЛИ В INT8")
    print("="*70)

    # 1. Загружаем или создаем модель
    if yaml_config_path and os.path.exists(yaml_config_path):
        print("1. Используется кастомизированная архитектура...")
        # На практике нужно загрузить обученную модель
        model = YOLO(model_path)
    else:
        print("1. Загрузка модели...")
        model = YOLO(model_path)

    # 2. Экспорт в ONNX с оптимизациями для квантизации
    print("\n2. Экспорт модели в ONNX с оптимизациями...")

    # Создаем директорию для экспорта если не существует
    export_dir = os.path.join(os.path.dirname(model_path), 'export')
    os.makedirs(export_dir, exist_ok=True)

    # Экспорт с параметрами для квантизации
    export_path = os.path.join(export_dir, 'model.onnx')

    model.export(
        format='onnx',
        imgsz=imgsz,
        simplify=True,
        opset=13,  # Opset 13+ для лучшей квантизации
        dynamic=False,  # Статические размеры
        half=False,  # Без FP16
        int8=False,  # Не использовать встроенную квантизацию
        device='cpu',  # Для совместимости
        workspace=4,  # GB
        nms=False,  # Отдельно для Cube.AI
    )

    # Находим экспортированный файл
    base_dir = os.path.dirname(model_path)
    onnx_files = glob.glob(os.path.join(base_dir, '*.onnx'))
    if not onnx_files:
        onnx_files = glob.glob(os.path.join(export_dir, '*.onnx'))

    if not onnx_files:
        print("ONNX файл не найден!")
        return None

    onnx_path = onnx_files[0]
    print(f"ONNX файл создан: {onnx_path}")

    # 3. ПРЕДВАРИТЕЛЬНАЯ ОБРАБОТКА ДЛЯ КВАНТИЗАЦИИ
    print("\n3. Предварительная обработка модели...")

    model_proto = onnx.load(onnx_path)

    # Оптимизация для квантизации
    from onnxruntime.quantization.shape_inference import quant_pre_process

    # Удаляем узлы, мешающие квантизации
    def remove_unsupported_nodes(model_proto):
        """Удаляет узлы, не поддерживающие INT8"""
        nodes_to_remove = []
        for i, node in enumerate(model_proto.graph.node):
            if node.op_type in ['Resize', 'Upsample']:
                # Заменяем Upsample на Resize с поддержкой INT8
                if node.op_type == 'Upsample':
                    # Создаем Resize узел
                    resize_node = helper.make_node(
                        'Resize',
                        inputs=node.input,
                        outputs=node.output,
                        name=node.name + '_resize',
                        mode='nearest',
                        nearest_mode='floor'
                    )
                    model_proto.graph.node[i].CopyFrom(resize_node)

        return model_proto

    model_proto = remove_unsupported_nodes(model_proto)

    # 4. АВТОМАТИЧЕСКАЯ КВАНТИЗАЦИЯ С КАЛИБРОВКОЙ
    print("\n4. Автоматическая квантизация с калибровкой...")

    try:
        from onnxruntime.quantization import (
            quantize_static,
            QuantFormat,
            QuantType,
            CalibrationMethod,
            create_calibrator,
            write_calibration_table
        )

        # Создаем реалистичные калибровочные данные для YOLO
        class YOLOCalibrationDataReader:
            def __init__(self, img_size=640, num_samples=20):
                self.img_size = img_size
                self.num_samples = num_samples
                self.current_idx = 0

                # Создаем реалистичные данные (нормализованные как в YOLO)
                self.data = []
                for i in range(num_samples):
                    # Изображение в формате YOLO: [1, 3, H, W], нормализованное
                    img = np.random.uniform(0, 1, (1, 3, img_size, img_size)).astype(np.float32)
                    self.data.append({'images': img})

                print(f"  Создано {num_samples} калибровочных изображений")

            def get_next(self):
                if self.current_idx < self.num_samples:
                    data = self.data[self.current_idx]
                    self.current_idx += 1
                    return data
                return None

            def rewind(self):
                self.current_idx = 0

        # Создаем калибровочный датасет
        calibration_data_reader = YOLOCalibrationDataReader(img_size=imgsz, num_samples=20)

        # Путь для квантизированной модели
        int8_path = onnx_path.replace('.onnx', '_int8_quantized.onnx')

        # Выполняем статическую квантизацию
        print("  Выполнение квантизации...")

        quantize_static(
            model_input=onnx_path,
            model_output=int8_path,
            calibration_data_reader=calibration_data_reader,
            quant_format=QuantFormat.QDQ,  # QDQ формат для Cube.AI
            activation_type=QuantType.QInt8,
            weight_type=QuantType.QInt8,
            per_channel=False,  # По слоям для совместимости
            reduce_range=False,
            nodes_to_quantize=[],  # Квантовать все узлы
            nodes_to_exclude=[],   # Не исключать никакие узлы
            calibrate_method=CalibrationMethod.MinMax,
            extra_options={
                'ActivationSymmetric': False,
                'WeightSymmetric': True,
                'EnableSubgraph': True,
                'ForceQuantizeNoInputCheck': True,
                'MatMulConstBOnly': False,
                'AddQDQPairToWeight': True,
                'OpTypesToExcludeOutputQuantization': [],
                'DedicatedQDQPair': True,
                'QDQOpTypePerChannelSupportToAxis': {
                    'Conv': 0,
                    'ConvTranspose': 0,
                    'MatMul': 0
                }
            }
        )

        print(f"Автоматическая квантизация завершена: {int8_path}")

    except Exception as e:
        print(f"Ошибка автоматической квантизации: {e}")
        print("  Используем ручную квантизацию...")
        int8_path = manual_quantization_fallback(onnx_path, imgsz)

    # 5. ПРИНУДИТЕЛЬНАЯ КОНВЕРТАЦИЯ ОСТАВШИХСЯ FP32
    print("\n5. Проверка и принудительная конвертация оставшихся FP32...")

    int8_path = force_int8_conversion(int8_path)

    # 6. ВАЛИДАЦИЯ И ВЕРИФИКАЦИЯ
    print("\n6. Валидация квантизированной модели...")

    validation_results = validate_quantized_model(int8_path, onnx_path)

    # 7. ОПТИМИЗАЦИЯ ДЛЯ CUBE.AI
    print("\n7. Оптимизация для STM32 Cube.AI...")

    final_path = optimize_for_cubeai(int8_path)

    # 8. ФИНАЛЬНЫЙ ОТЧЕТ
    print("\n" + "="*70)
    print("ФИНАЛЬНЫЙ ОТЧЕТ")
    print("="*70)

    print_report(onnx_path, final_path)

    return final_path

def manual_quantization_fallback(onnx_path, imgsz):
    """Резервный метод ручной квантизации"""
    print("  Запуск ручной квантизации...")

    model = onnx.load(onnx_path)
    int8_path = onnx_path.replace('.onnx', '_manual_int8.onnx')

    # 1. Конвертируем все веса в INT8
    for initializer in model.graph.initializer:
        if initializer.data_type == onnx.TensorProto.FLOAT:
            # Конвертируем данные
            float_data = numpy_helper.to_array(initializer)

            # Находим масштаб для квантизации
            abs_max = np.max(np.abs(float_data))
            if abs_max == 0:
                scale = 1.0
            else:
                scale = 127.0 / abs_max

            # Квантуем
            int8_data = np.clip(np.round(float_data * scale), -128, 127).astype(np.int8)

            # Обновляем тензор
            initializer.data_type = onnx.TensorProto.INT8
            initializer.raw_data = int8_data.tobytes()

            # Добавляем узел DequantizeLinear
            scale_name = f"{initializer.name}_scale"
            zero_point_name = f"{initializer.name}_zero_point"

            # Создаем scale и zero_point
            scale_tensor = helper.make_tensor(
                scale_name,
                onnx.TensorProto.FLOAT,
                [1],
                [1.0 / scale]  # Обратный scale для деквантизации
            )

            zero_point_tensor = helper.make_tensor(
                zero_point_name,
                onnx.TensorProto.INT8,
                [1],
                [0]
            )

            # Добавляем тензоры в модель
            model.graph.initializer.append(scale_tensor)
            model.graph.initializer.append(zero_point_tensor)

            # Находим узел, использующий этот тензор
            for node in model.graph.node:
                if initializer.name in node.input:
                    # Заменяем вход на квантованный
                    dq_node = helper.make_node(
                        'DequantizeLinear',
                        inputs=[initializer.name, scale_name, zero_point_name],
                        outputs=[initializer.name + '_dequant'],
                        name=f'Dequantize_{initializer.name}'
                    )
                    # Обновляем вход узла
                    idx = node.input.index(initializer.name)
                    node.input[idx] = initializer.name + '_dequant'

                    # Вставляем DQ узел перед текущим узлом
                    model.graph.node.insert(list(model.graph.node).index(node), dq_node)

    # 2. Добавляем QuantizeLinear для входов
    input_name = model.graph.input[0].name
    q_input = helper.make_node(
        'QuantizeLinear',
        inputs=[input_name, 'input_scale', 'input_zero_point'],
        outputs=[input_name + '_quantized'],
        name='QuantizeInput'
    )

    # Добавляем scale и zero_point для входа
    input_scale = helper.make_tensor(
        'input_scale',
        onnx.TensorProto.FLOAT,
        [1],
        [1.0 / 127.0]  # Предполагаем диапазон [-1, 1]
    )

    input_zero_point = helper.make_tensor(
        'input_zero_point',
        onnx.TensorProto.INT8,
        [1],
        [0]
    )

    model.graph.initializer.extend([input_scale, input_zero_point])

    # Вставляем QuantizeLinear в начало
    model.graph.node.insert(0, q_input)

    # Обновляем входы всех узлов, которые используют оригинальный вход
    for node in model.graph.node:
        if input_name in node.input:
            idx = node.input.index(input_name)
            node.input[idx] = input_name + '_quantized'

    # 3. Добавляем DequantizeLinear для выходов
    for output in model.graph.output:
        dq_output = helper.make_node(
            'DequantizeLinear',
            inputs=[output.name, 'output_scale', 'output_zero_point'],
            outputs=[output.name + '_dequantized'],
            name=f'Dequantize_{output.name}'
        )

        output_scale = helper.make_tensor(
            'output_scale',
            onnx.TensorProto.FLOAT,
            [1],
            [1.0]
        )

        output_zero_point = helper.make_tensor(
            'output_zero_point',
            onnx.TensorProto.INT8,
            [1],
            [0]
        )

        model.graph.initializer.extend([output_scale, output_zero_point])
        model.graph.node.append(dq_output)

        # Обновляем выход
        output.name = output.name + '_dequantized'

    # Сохраняем модель
    onnx.save(model, int8_path)
    return int8_path

def force_int8_conversion(model_path):
    """Принудительная конвертация всех оставшихся FP32 параметров"""
    print("  Принудительная конвертация оставшихся FP32...")

    model = onnx.load(model_path)

    fp32_count = 0
    int8_count = 0

    for initializer in model.graph.initializer:
        if initializer.data_type == onnx.TensorProto.FLOAT:
            fp32_count += 1

            # Принудительно конвертируем в INT8
            try:
                float_data = numpy_helper.to_array(initializer)
                # Простая квантизация
                abs_max = np.max(np.abs(float_data))
                scale = 127.0 / abs_max if abs_max > 0 else 1.0
                int8_data = np.clip(np.round(float_data * scale), -128, 127).astype(np.int8)

                initializer.data_type = onnx.TensorProto.INT8
                initializer.raw_data = int8_data.tobytes()
                int8_count += 1

            except:
                # Если не получается, заменяем нулями
                shape = list(initializer.dims)
                int8_data = np.zeros(shape, dtype=np.int8)
                initializer.data_type = onnx.TensorProto.INT8
                initializer.raw_data = int8_data.tobytes()
                int8_count += 1

    if fp32_count > 0:
        print(f"  Конвертировано {fp32_count} FP32 тензоров в INT8")

        # Сохраняем с новым именем
        forced_path = model_path.replace('.onnx', '_forced_int8.onnx')
        onnx.save(model, forced_path)
        return forced_path

    return model_path

def validate_quantized_model(int8_path, original_onnx_path):
    """Валидация квантизированной модели"""
    print("  Валидация квантизированной модели...")

    # Загружаем модели
    int8_model = onnx.load(int8_path)

    # Статистика по типам
    type_stats = {}
    total_params = 0

    for initializer in int8_model.graph.initializer:
        dtype = initializer.data_type
        dtype_name = onnx.TensorProto.DataType.Name(dtype)
        num_params = np.prod(initializer.dims) if initializer.dims else 1

        type_stats[dtype_name] = type_stats.get(dtype_name, 0) + num_params
        total_params += num_params

    # Отчет
    print("\n  СТАТИСТИКА ТИПОВ ДАННЫХ:")
    for dtype, count in type_stats.items():
        percentage = (count / total_params) * 100
        print(f"    {dtype}: {count:,} параметров ({percentage:.1f}%)")

    # Проверяем наличие FP32
    if 'FLOAT' in type_stats:
        print(f"  Обнаружены FP32 параметры: {type_stats['FLOAT']:,}")
    else:
        print("  Все параметры в INT8!")

    return type_stats

def optimize_for_cubeai(model_path):
    """Оптимизация модели для STM32 Cube.AI"""
    print("  Оптимизация для Cube.AI...")

    model = onnx.load(model_path)
    optimized_path = model_path.replace('.onnx', '_cubeai.onnx')

    # 1. Удаляем лишние узлы для Cube.AI
    nodes_to_remove = []
    for i, node in enumerate(model.graph.node):
        # Cube.AI лучше работает без некоторых узлов
        if node.op_type in ['Identity', 'Dropout']:
            nodes_to_remove.append(i)

    # Удаляем в обратном порядке
    for i in sorted(nodes_to_remove, reverse=True):
        del model.graph.node[i]

    # 2. Убеждаемся, что все имена корректны
    for i, node in enumerate(model.graph.node):
        if not node.name or node.name.startswith('Unnamed'):
            node.name = f'node_{i}'

    # 3. Упрощаем граф
    try:
        model = shape_inference.infer_shapes(model)
    except:
        pass

    # 4. Сохраняем оптимизированную модель
    onnx.save(model, optimized_path)

    print(f"  Оптимизированная модель сохранена: {optimized_path}")
    return optimized_path

def print_report(original_path, quantized_path):
    """Печать финального отчета"""
    import os

    # Размеры файлов
    original_size = os.path.getsize(original_path) / 1024 / 1024
    quantized_size = os.path.getsize(quantized_path) / 1024 / 1024

    print(f"\nСРАВНЕНИЕ РАЗМЕРОВ:")
    print(f"  Оригинальная модель (FP32): {original_size:.2f} MB")
    print(f"  Квантизированная модель (INT8): {quantized_size:.2f} MB")
    print(f"  Сжатие: {(1 - quantized_size/original_size)*100:.1f}%")

    print(f"\nФАЙЛЫ:")
    print(f"  Оригинал: {original_path}")
    print(f"  INT8: {quantized_path}")

    print("\nИНСТРУКЦИЯ ДЛЯ STM32 CUBE.AI:")
    print("  1. Откройте STM32CubeMX с вашим проектом")
    print("  2. Перейдите в Software Packs → STMicroelectronics.X-CUBE-AI")
    print("  3. Нажмите 'Add Network' или 'Import Model'")
    print("  4. Выберите файл квантизированной модели")
    print("  5. В настройках импорта:")
    print("     - Формат: ONNX")
    print("     - Квантизация: INT8")
    print("     - Input scaling: Проверьте нормализацию")
    print("  6. Настройте выделение памяти")
    print("  7. Сгенерируйте код и протестируйте")

    print("\nВАЖНЫЕ ЗАМЕЧАНИЯ:")
    print("  - Cube.AI требует точной нормализации входных данных")
    print("  - Убедитесь, что входные данные масштабированы так же, как при обучении")
    print("  - Для кастомизированных моделей проверьте совместимость операторов")
    print("  - Рекомендуется использовать batch_size=1 для микроконтроллеров")

# УТИЛИТЫ ДЛЯ ПРОВЕРКИ
def check_model_compatibility(model_path):
    """Проверка совместимости модели с Cube.AI"""
    print("\n" + "="*70)
    print("ПРОВЕРКА СОВМЕСТИМОСТИ С CUBE.AI")
    print("="*70)

    model = onnx.load(model_path)

    # Список поддерживаемых операторов Cube.AI
    supported_ops = [
        'Conv', 'Add', 'Mul', 'Relu', 'Clip', 'Sigmoid', 'MaxPool',
        'AveragePool', 'GlobalAveragePool', 'Reshape', 'Flatten',
        'Concat', 'Resize', 'Transpose', 'Split', 'Slice', 'Gather'
    ]

    # Проверяем операторы
    unsupported_ops = []
    for node in model.graph.node:
        if node.op_type not in supported_ops:
            unsupported_ops.append(node.op_type)

    if unsupported_ops:
        print(f"Неподдерживаемые операторы: {set(unsupported_ops)}")
        print("  Cube.AI может не поддерживать эти операторы.")
        print("  Рассмотрите возможность замены или упрощения модели.")
    else:
        print("Все операторы поддерживаются Cube.AI!")

    # Проверка входов/выходов
    print(f"\nВходы модели: {[i.name for i in model.graph.input]}")
    print(f"Выходы модели: {[o.name for o in model.graph.output]}")

    # Рекомендации
    print("\nРЕКОМЕНДАЦИИ ДЛЯ CUBE.AI:")
    print("  1. Имя входа должно быть 'images' или 'input'")
    print("  2. Имя выхода должно быть понятным (output0, output1, etc.)")
    print("  3. Размерность входа: [1, 3, H, W]")
    print("  4. Избегайте динамических размерностей")

# ОСНОВНАЯ ФУНКЦИЯ
def quantize_yolo_for_stm32(model_path, yaml_config_path=None, imgsz=640):
    """
    Основная функция квантизации YOLO для STM32

    Args:
        model_path: Путь к файлу .pt модели
        yaml_config_path: Путь к YAML конфигу (опционально)
        imgsz: Размер изображения

    Returns:
        Путь к квантизированной модели
    """
    print("ЗАПУСК КВАНТИЗАЦИИ ДЛЯ STM32")
    print(f"Модель: {model_path}")
    if yaml_config_path:
        print(f"Конфиг: {yaml_config_path}")
    print(f"Размер изображения: {imgsz}")
    print("="*70)

    try:
        # Выполняем квантизацию
        quantized_model = quantize_custom_yolo_to_int8(
            model_path,
            yaml_config_path,
            imgsz
        )

        if quantized_model:
            # Проверяем совместимость
            check_model_compatibility(quantized_model)

            print("\n" + "="*70)
            print("КВАНТИЗАЦИЯ ЗАВЕРШЕНА УСПЕШНО!")
            print("="*70)

            return quantized_model
        else:
            print("\nОШИБКА КВАНТИЗАЦИИ!")
            return None

    except Exception as e:
        print(f"\nКРИТИЧЕСКАЯ ОШИБКА: {e}")
        import traceback
        traceback.print_exc()
        return None

In [4]:
YAML_CONFIG = ''  # Укажите путь к вашему YAML

# Запускаем квантизацию
quantized_model_path = quantize_yolo_for_stm32(
    model_path=MODEL_PATH,
    yaml_config_path=YAML_CONFIG if os.path.exists(YAML_CONFIG) else None,
    imgsz=640
)

if quantized_model_path:
    print(f"\nМодель успешно квантизирована!")
    print(f"Файл: {quantized_model_path}")
    print("\nТеперь вы можете импортировать эту модель в STM32 Cube.AI")
else:
    print("\nНе удалось выполнить квантизацию")

🚀 ЗАПУСК КВАНТИЗАЦИИ ДЛЯ STM32
Модель: models/YOLOv8n_augmented/weights/best.pt
Размер изображения: 640
ПОЛНАЯ КВАНТИЗАЦИЯ КАСТОМИЗИРОВАННОЙ YOLOv8 МОДЕЛИ В INT8
1. Загрузка модели...

2. Экспорт модели в ONNX с оптимизациями...
Ultralytics 8.3.239  Python-3.13.9 torch-2.7.1+cu118 CPU (12th Gen Intel Core(TM) i5-12400F)
Model summary (fused): 72 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'models\YOLOv8n_augmented\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 11, 8400) (11.8 MB)

ONNX: starting export with onnx 1.20.0 opset 13...
ONNX: slimming with onnxslim 0.1.80...
ONNX: export success  1.1s, saved as 'models\YOLOv8n_augmented\weights\best.onnx' (11.7 MB)

Export complete (1.3s)
Results saved to C:\Users\1\Documents\Stud BMSTU\8 semestr\VKR\VKR_project\models\YOLOv8n_augmented\weights
Predict:         yolo predict task=detect model=models\YOLOv8n_augmented\weights\best.onnx imgsz=640  
Validate:        yolo val task

  Создано 20 калибровочных изображений
  Выполнение квантизации...


✓ Автоматическая квантизация завершена: models/YOLOv8n_augmented/weights\best_int8_quantized.onnx

5. Проверка и принудительная конвертация оставшихся FP32...
  Принудительная конвертация оставшихся FP32...
  Конвертировано 410 FP32 тензоров в INT8

6. Валидация квантизированной модели...
  Валидация квантизированной модели...

  СТАТИСТИКА ТИПОВ ДАННЫХ:
    INT8: 3,044,224 параметров (99.8%)
    INT64: 24 параметров (0.0%)
    INT32: 5,476 параметров (0.2%)
  ✅ Все параметры в INT8!

7. Оптимизация для STM32 Cube.AI...
  Оптимизация для Cube.AI...
  Оптимизированная модель сохранена: models/YOLOv8n_augmented/weights\best_int8_quantized_forced_int8_cubeai.onnx

ФИНАЛЬНЫЙ ОТЧЕТ

📊 СРАВНЕНИЕ РАЗМЕРОВ:
  Оригинальная модель (FP32): 11.70 MB
  Квантизированная модель (INT8): 3.31 MB
  Сжатие: 71.7%

📁 ФАЙЛЫ:
  Оригинал: models/YOLOv8n_augmented/weights\best.onnx
  INT8: models/YOLOv8n_augmented/weights\best_int8_quantized_forced_int8_cubeai.onnx

🎯 ИНСТРУКЦИЯ ДЛЯ STM32 CUBE.AI:
  1. Открой

In [10]:
# =========================
# 4. ПРОВЕРКА КВАНТИЗАЦИИ
# =========================

def check_int8_quantization(onnx_path):
    """Быстрая проверка квантизации"""

    import onnx
    import os

    model = onnx.load(onnx_path)

    int8_bytes = 0
    fp32_bytes = 0
    total_params = 0

    print("Проверка квантизации INT8:")
    print("-"*50)

    for tensor in model.graph.initializer:
        # Размерность
        shape = []
        if hasattr(tensor, 'dims'):
            for dim in tensor.dims:
                if isinstance(dim, int):
                    shape.append(dim)
                elif hasattr(dim, 'dim_value'):
                    shape.append(dim.dim_value)

        if shape:
            params = 1
            for dim in shape:
                params *= dim
            total_params += params

            # Байты
            if tensor.data_type == 3:  # INT8
                int8_bytes += params * 1  # 1 байт на параметр
            elif tensor.data_type == 1:  # FP32
                fp32_bytes += params * 4  # 4 байта на параметр

    print(f"Всего параметров: {total_params:,}")
    print(f"INT8 параметров: {int8_bytes:,} байт ({int8_bytes/1024:.1f} KB)")
    print(f"FP32 параметров: {fp32_bytes:,} байт ({fp32_bytes/1024:.1f} KB)")

    if fp32_bytes > 0:
        print(f"\nПроблема: {fp32_bytes/1024:.1f} KB все еще в FP32!")
        print("Модель не полностью квантизирована.")
    else:
        print(f"\nМодель полностью квантизирована в INT8")
        print(f"Размер на МК: ~{int8_bytes/1024:.0f} KB")

    return int8_bytes, fp32_bytes

# Быстрая проверка
check_int8_quantization(ONNX_WEIGHTS_PATH)

Проверка квантизации INT8:
--------------------------------------------------
Всего параметров: 3,049,104
INT8 параметров: 3,043,667 байт (2972.3 KB)
FP32 параметров: 0 байт (0.0 KB)

✅ Модель полностью квантизирована в INT8
Размер на МК: ~2972 KB


(3043667, 0)